# FacTora — Multimodal Image Deepfake & AI Generation Detector Training
**Project**: FacTora Multimodal Credibility Intelligence Platform  
**Author**: Pranav Lamkhade  
**Architecture**: Vision Transformer / ResNet / ConvNeXt for Image Classification  
**Task**: Binary Classification — `REAL (0)` vs `AI-GENERATED / FAKE (1)`  
**Benchmark Dataset**: CIFAKE (120,000 images: 60k Real from CIFAR-10, 60k Synthetic from Stable Diffusion)  
**Target Accuracy**: >95.0% Validation Accuracy  
**Output**: Lightweight `model.safetensors` (~20-80 MB) for on-demand inference in FacTora.

In [ ]:
# Step 1: Environment & GPU Hardware Check
import torch
import torchvision
import transformers

print(f"PyTorch Version      : {torch.__version__}")
print(f"CUDA Available       : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU Device Name      : {torch.cuda.get_device_name(0)}")
    print(f"Device Memory        : {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
    print(f"BF16/FP16 Supported  : {torch.cuda.is_bf16_supported()}")
print(f"Transformers Version : {transformers.__version__}")

In [ ]:
# Step 2: Dependencies Installation
!pip install -q evaluate datasets accelerate safetensors

In [ ]:
# Step 3: Imports & Configuration
import os
import glob
import json
import numpy as np
from PIL import Image
import evaluate
import torch
from torch.utils.data import Dataset
from torchvision import transforms
from transformers import (
    AutoImageProcessor,
    AutoModelForImageClassification,
    TrainingArguments,
    Trainer,
    DefaultDataCollator
)

# ── Training Configuration ──────────────────────────────────────────────────
BASE_MODEL_NAME = "microsoft/resnet-50"  # or 'google/vit-base-patch16-224-in21k'
OUTPUT_DIR = "/kaggle/working/factora_image_model"
BATCH_SIZE = 64
NUM_EPOCHS = 4
LEARNING_RATE = 5e-5
IMAGE_SIZE = (224, 224)
ID2LABEL = {0: "REAL", 1: "FAKE"}
LABEL2ID = {"REAL": 0, "FAKE": 1}

os.makedirs(OUTPUT_DIR, exist_ok=True)

In [ ]:
# Step 4: Locate Dataset on Kaggle
# CIFAKE is commonly mounted at /kaggle/input/cifake-real-and-ai-generated-synthetic-images
dataset_candidates = [
    "/kaggle/input/cifake-real-and-ai-generated-synthetic-images",
    "/kaggle/input/cifake",
    "/kaggle/input/140k-real-and-fake-faces"
]

DATASET_PATH = None
for path in dataset_candidates:
    if os.path.exists(path):
        DATASET_PATH = path
        break

if DATASET_PATH:
    print(f"Found dataset at: {DATASET_PATH}")
else:
    print("Warning: CIFAKE dataset not found in default path. Please attach CIFAKE via Kaggle Add Data panel.")
    DATASET_PATH = "/kaggle/input/cifake-real-and-ai-generated-synthetic-images"

In [ ]:
# Step 5: PyTorch Dataset Loader with On-The-Fly Augmentation
class CIFAKEDataset(Dataset):
    def __init__(self, root_dir, split="train", transform=None):
        self.image_paths = []
        self.labels = []
        self.transform = transform
        
        split_dir = os.path.join(root_dir, split)
        real_dir = os.path.join(split_dir, "REAL")
        fake_dir = os.path.join(split_dir, "FAKE")
        
        if os.path.exists(real_dir) and os.path.exists(fake_dir):
            for f in glob.glob(os.path.join(real_dir, "*.*")):
                self.image_paths.append(f)
                self.labels.append(0)  # REAL
            for f in glob.glob(os.path.join(fake_dir, "*.*")):
                self.image_paths.append(f)
                self.labels.append(1)  # FAKE
        
        print(f"Loaded {len(self.image_paths)} samples for split: {split} (Real: {self.labels.count(0)}, Fake: {self.labels.count(1)})")

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        path = self.image_paths[idx]
        label = self.labels[idx]
        image = Image.open(path).convert("RGB")
        
        if self.transform:
            image = self.transform(image)
            
        return {"pixel_values": image, "labels": torch.tensor(label, dtype=torch.long)}

# Image Preprocessor transforms
train_transforms = transforms.Compose([
    transforms.Resize(IMAGE_SIZE),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(brightness=0.1, contrast=0.1, saturation=0.1),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

val_transforms = transforms.Compose([
    transforms.Resize(IMAGE_SIZE),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

train_dataset = CIFAKEDataset(DATASET_PATH, split="train", transform=train_transforms)
val_dataset = CIFAKEDataset(DATASET_PATH, split="test", transform=val_transforms)

In [ ]:
# Step 6: Load Image Processor and Pretrained Model
processor = AutoImageProcessor.from_pretrained(BASE_MODEL_NAME)
model = AutoModelForImageClassification.from_pretrained(
    BASE_MODEL_NAME,
    num_labels=2,
    id2label=ID2LABEL,
    label2id=LABEL2ID,
    ignore_mismatched_sizes=True
)

total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Model Architecture : {BASE_MODEL_NAME}")
print(f"Total Parameters   : {total_params / 1e6:.2f} M")
print(f"Trainable Params   : {trainable_params / 1e6:.2f} M")

In [ ]:
# Step 7: Metrics Function
acc_metric = evaluate.load("accuracy")
f1_metric = evaluate.load("f1")
precision_metric = evaluate.load("precision")
recall_metric = evaluate.load("recall")

def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    preds = np.argmax(predictions, axis=1)
    
    accuracy = acc_metric.compute(predictions=preds, references=labels)["accuracy"]
    f1 = f1_metric.compute(predictions=preds, references=labels, average="weighted")["f1"]
    precision = precision_metric.compute(predictions=preds, references=labels, average="weighted")["precision"]
    recall = recall_metric.compute(predictions=preds, references=labels, average="weighted")["recall"]
    
    return {
        "accuracy": accuracy,
        "f1": f1,
        "precision": precision,
        "recall": recall
    }

In [ ]:
# Step 8: Training Arguments & Trainer Setup
training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    evaluation_strategy="epoch",
    save_strategy="epoch",
    learning_rate=LEARNING_RATE,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    num_train_epochs=NUM_EPOCHS,
    weight_decay=0.01,
    warmup_ratio=0.1,
    logging_steps=100,
    load_best_model_at_end=True,
    metric_for_best_model="accuracy",
    fp16=torch.cuda.is_available(),
    save_total_limit=1,
    report_to="none"
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    tokenizer=processor,
    compute_metrics=compute_metrics,
    data_collator=DefaultDataCollator()
)

print("Trainer initialized. Ready for training.")

In [ ]:
# Step 9: Execute Training Loop
print("Starting training on Kaggle GPU...")
train_results = trainer.train()
print("Training Complete!")
print(train_results)

In [ ]:
# Step 10: Final Evaluation on Test Split
eval_results = trainer.evaluate()
print("Final Evaluation Metrics on Test Set:")
for k, v in eval_results.items():
    print(f"  {k}: {v}")

In [ ]:
# Step 11: Export Model Weights and Config for FacTora
final_model_dir = os.path.join(OUTPUT_DIR, "factora_image_model")
trainer.save_model(final_model_dir)
processor.save_pretrained(final_model_dir)

print(f"Model and preprocessor saved to: {final_model_dir}")
print("Files generated:")
for f in os.listdir(final_model_dir):
    fpath = os.path.join(final_model_dir, f)
    size_mb = os.path.getsize(fpath) / 1e6
    print(f"  - {f} ({size_mb:.2f} MB)")

In [ ]:
# Step 12: Interactive Single-Image Forensic Test
def verify_test_image(image_path):
    img = Image.open(image_path).convert("RGB")
    inputs = processor(img, return_tensors="pt").to(model.device)
    with torch.no_grad():
        logits = model(**inputs).logits
        probs = torch.softmax(logits, dim=1)[0].cpu().numpy()
    
    pred_idx = np.argmax(probs)
    verdict = ID2LABEL[pred_idx]
    confidence = float(probs[pred_idx]) * 100
    print(f"VERDICT: {verdict} | CONFIDENCE: {confidence:.2f}% | (Real: {probs[0]*100:.1f}%, Fake: {probs[1]*100:.1f}%)")

# Test on sample image if available
if len(val_dataset) > 0:
    sample_path = val_dataset.image_paths[0]
    verify_test_image(sample_path)